# Computation of Granularity Adjustments for the IDB Portfolio with EEAs and B- Floor

In [35]:
import numpy as np
from scipy.optimize import fsolve, minimize
import matplotlib.pyplot as plt 
import pandas as pd
import copy
from tqdm import tqdm
%load_ext autoreload
%autoreload 2
%aimport IS_misc_functions
from IS_misc_functions import *
from misc_functions import *
from actuarial_training import *
from MtM_training import *
from scipy.optimize import minimize

import warnings
warnings.filterwarnings("ignore")

# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers
# from tensorflow.keras import regularizers
# from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler,normalize
# from sklearn.model_selection import train_test_split
# from sklearn.utils import shuffle

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Create Portfolios

Read in everything

In [36]:
by_country = pd.read_excel("MDB portfolios.xlsx", sheet_name='by country')
maturity = pd.read_excel("MDB portfolios.xlsx", sheet_name='maturity')
summary = pd.read_excel("MDB portfolios.xlsx", sheet_name='summary of current info')
comments = pd.read_excel("MDB portfolios.xlsx", sheet_name='comments')
rating = pd.read_excel("MDB portfolios.xlsx", sheet_name='rating')
transition_matrix_SP = pd.read_csv("transition_matrix_SP.csv",delimiter = ",",index_col = 0)
transition_matrix_RC = pd.read_csv("transition_matrix_RC.csv",delimiter = ";",index_col = 0)

Preprocessing the Data

### Display the S & P Transition Matrix

In [37]:
SP_dict = trans_matrix_to_dict(transition_matrix_SP)
transition_matrix_SP

,AAA,AA+,AA,AA-,A+,A,A-,BBB+,BBB,BBB-,BB+,BB,BB-,B+,B,B-,Cs,D
AAA,96.79,2.71,0.42,0.00,0.00,0.00,0.01,0.00,0.00,0.00,0.00,0.07,0.00,0.00,0.00,0.00,0.00,0.00
AA+,6.45,85.16,6.61,1.77,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
AA,0.00,6.22,85.17,6.74,0.52,0.42,0.10,0.52,0.00,0.00,0.00,0.31,0.00,0.00,0.00,0.00,0.00,0.00
AA-,0.00,0.00,7.82,83.45,7.16,0.17,0.50,0.17,0.00,0.17,0.44,0.00,0.00,0.11,0.00,0.00,0.00,0.00
A+,0.00,0.00,0.07,13.35,73.28,9.30,2.03,1.12,0.14,0.63,0.07,0.00,0.00,0.00,0.00,0.00,0.00,0.01
A,0.00,0.00,0.00,1.15,12.33,77.29,5.71,1.68,0.77,0.96,0.10,0.00,0.00,0.00,0.00,0.00,0.00,0.01
A-,0.00,0.00,0.00,0.00,0.94,11.47,77.82,6.94,0.41,1.57,0.67,0.16,0.00,0.00,0.00,0.00,0.00,0.02
BBB+,0.00,0.00,0.00,0.00,0.00,2.16,12.39,70.86,11.24,2.41,0.60,0.24,0.06,0.00,0.00,0.00,0.00,0.04
BBB,0.00,0.00,0.00,0.00,0.00,0.00,1.87,16.60,68.05,11.16,0.99,0.11,0.00,0.50,0.22,0.11,0.33,0.06
BBB-,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.93,14.94,74.69,6.50,2.13,0.27,0.08,0.15,0.12,0.08,0.11


Create Dictionaries to assign PDs to the considered countries

In [38]:
#Small Tolerance to avoid 0s
tolerance = 10e-10

# list of banks
bank_names = list(summary.iloc[:,0])

# Exclude EIB and NIB from being considered
bank_names = [b for b in bank_names if b != "EIB" and b!= "NIB"]


# SP Rating of Countries
country_sp = {rating["Country"][i].lower().replace(" ", ""): 
              str(rating["sp_cb"][i]).replace(" ", "")  for i in range(len(rating["sp_cb"]))}

def create_PD_dict(transition_matrix,unrated_b_minus = True):
    country_dict = country_sp #copy.copy(country_sp)

    # The PDs of Rating Classes
    Rating_PD = {transition_matrix.index[i]: np.maximum(transition_matrix["D"][i],tolerance)  for i in range(len(transition_matrix.index))}

    # Companies with D rating are not in default wrt other MDBs:
    Rating_PD["D"] = Rating_PD["Cs"]
    Rating_PD["SD"] = Rating_PD["Cs"]
    
    if unrated_b_minus:
        for country_rating in set(country_dict.values()):
            if country_rating in ["C","CC","CCC-","CCC","CCC+"]:
                Rating_PD[country_rating] = Rating_PD["Cs"]
            if country_rating in ['0',0]:
                Rating_PD[country_rating] = Rating_PD["B-"] 

#         for country_rating in set(country_dict.values()):
#             if country_rating not in Rating_PD.keys():
#                 Rating_PD[country_rating] = "nan"

        for country in country_dict:
            if country_dict[country] in ["C","CC","CCC-","CCC","CCC+"]:
                country_dict[country] = "Cs"
            if country_dict[country] in ['0',0]:
                country_dict[country] = "B-"
    
    elif unrated_b_minus == False:

        # Adjust the PDs further 
        for country_rating in set(country_dict.values()):
            if country_rating in ["C","CC",'0',0,"CCC-","CCC","CCC+"]:
                Rating_PD[country_rating] = Rating_PD["Cs"] 

        for country_rating in set(country_dict.values()):
            if country_rating not in Rating_PD.keys():
                Rating_PD[country_rating] = "nan"

        for country in country_dict:
            if country_dict[country] in ["C","CC",'0',0,"CCC-","CCC","CCC+"]:
                country_dict[country] = "Cs"

    # Assign the ratings to the countries
    countries_PD = {country.lower().replace(" ", ""): Rating_PD[country_dict[country]] for country in country_dict.keys()}
    return countries_PD

In [39]:
countries_PD_SP = create_PD_dict(transition_matrix_SP)

Find the spaces between banks (in the excel sheet)

In [40]:
spaces = np.where(pd.isnull(by_country.iloc[:,3]))[0]
spaces[0] = 1
spaces = np.append(spaces,len(by_country.iloc[:,3]))

### Create the portfolios

Define a function to create the portfolios

In [41]:
bank_names_restricted = ['IDB']
spaces = np.where(pd.isnull(by_country.iloc[:,3]))[0]
spaces[0] = 1
spaces = np.append(spaces,len(by_country.iloc[:,3]))

In [42]:
def create_portfolio(q = 0.99, ELGD_val = 0.1,
                     transition_matrix = transition_matrix_SP,
                     countries_PD = countries_PD_SP,
                    constant_rho = False,
                    rho = 0.35,
                    normalize_shares = True):
    portfolios = {}
    for i in range(len(bank_names)):
        considered_countries = np.array(by_country.iloc[:,1][(spaces[i]+1):(spaces[i+1]-1)])
        PD = []
        EAD = []
        g = []
        for j in range(len(considered_countries)):
            country = considered_countries[j].lower().replace(" ", "")
            if country in countries_PD.keys() and countries_PD[country] != 'nan': # nur anlegen wenn existent
                PD.append(countries_PD[country] /100)
                EAD.append(by_country.iloc[:,3][(spaces[i]+1)+j])
                g.append(S-list(transition_matrix.index).index(country_sp[country]))
        if normalize_shares:
            EAD = EAD/np.sum(np.array(EAD,dtype = np.float64))
        PD = np.array(PD)
        ELGD = np.array([ELGD_val]*len(EAD))
        if constant_rho:
            def to_solve(om):
                b = (0.11852-0.05478*np.log(PD))**2
                R_corr = rho
                K = ((ELGD)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD*ELGD)
                alpha_X =  gamma.ppf(q,a =0.25, scale = 1/0.25)
                return  (np.abs(ELGD*PD*om*(alpha_X-1)-K))
        else:
            def to_solve(om):
                b = (0.11852-0.05478*np.log(PD))**2
                R_corr = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50)))
                K = ((ELGD)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD*ELGD)
                alpha_X =  gamma.ppf(q,a =0.25, scale = 1/0.25)
                return  (np.abs(ELGD*PD*om*(alpha_X-1)-K))
        omega = fsolve(to_solve,[0.25]*len(EAD))
        omega = np.minimum(omega,1)
        #bnds = [(0,1) for i in range(len(EAD))]
        if bank_names[i] in bank_names_restricted:
            portfolios[bank_names[i]] = [EAD,ELGD,PD,omega,g]
    return portfolios

Create the portfolios for different quantile levels and different transiiton matricies under different assumptions

In [43]:
portfolios_0999 = create_portfolio(q = 0.999,transition_matrix = transition_matrix_SP,countries_PD = countries_PD_SP)

duplicate the portfolio

In [44]:
portfolios_0999['IDB_B-'] = copy.deepcopy(portfolios_0999['IDB'])

apply the EEA

In [46]:
Countries_EEA = [
    "Argentina", "Bahamas", "Barbados", "Belize", "Bolivia", "Brazil", "Chile",
    "Colombia", "Costa Rica", "Dominican Republic", "Ecuador", "El Salvador",
    "Guatemala", "Guyana", "Haiti", "Honduras", "Jamaica", "Mexico", "Nicaragua",
    "Panama", "Paraguay", "Peru", "Suriname", "Trinidad and Tobago", "Uruguay",
    "Venezuela", "Angola", "Armenia", "Bangladesh", "Bosnia and Herzegovina", "Egypt",
    "Georgia", "India", "Indonesia", "Jordan", "Macedonia", "Montenegro", "Morocco",
    "Nigeria", "Pakistan", "Servia", "Sri Lanka", "Tunisia", "Turkey", "Vietnam"
]
EEA = [
    14645, 725, 629, 157, 3920, 13390, 2224, 10253, 2411, 3511,
    6235, 2085, 1913, 787, 0, 3068, 1692, 14210, 2316, 4150,
    3071, 3158, 656, 532, 3370, 2011, 85, 118, 673, 99,
    720, 97, 525, 885, 144, 130, 116, 990, 95, 977,
    195, 48, 990, 311, 203
]

Create Entry for IDB subject to EEAs

In [47]:
PD_vector = []
EAD_vector = []
g_vector = []

for j in range(len(Countries_EEA)):
        country = Countries_EEA[j].lower().replace(" ", "")
        if country in countries_PD_SP.keys() and countries_PD_SP[country] != 'nan': # nur anlegen wenn existent
            PD_vector.append(countries_PD_SP[country] /100)
            EAD_vector.append(EEA[j])
            g_vector.append(S-list(transition_matrix_SP["D"]).index(countries_PD_SP[country]))
EAD_vector = np.array(EAD_vector)/np.sum(EAD_vector)
ELGD_vector = np.array([0.1]*len(EAD_vector))
PD_vector = np.array(PD_vector)
q=0.999
def to_solve(om):
    b = (0.11852-0.05478*np.log(PD_vector))**2
    R_corr = 0.12*(1-np.exp(-50*PD_vector))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD_vector))/(1-np.exp(-50)))
    K = ((ELGD_vector)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD_vector)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD_vector*ELGD_vector)
    alpha_X =  gamma.ppf(q,a =0.25, scale = 1/0.25)
    return  (np.abs(ELGD_vector*PD_vector*om*(alpha_X-1)-K))
omega_vector = fsolve(to_solve,[0.25]*len(EAD_vector))
portfolios_0999['IDB_EEA'] = [EAD_vector,ELGD_vector,PD_vector,omega_vector,g_vector]

Create Entry for IDB subject to EEAs & B- Floor

In [48]:
PD_vector = []
EAD_vector = []
g_vector = []

for j in range(len(Countries_EEA)):
        country = Countries_EEA[j].lower().replace(" ", "")
        if country in countries_PD_SP.keys() and countries_PD_SP[country] != 'nan': # nur anlegen wenn existent
            PD_vector.append(min(countries_PD_SP[country] /100,7.590e-02))
            EAD_vector.append(EEA[j])
            g_vector.append(max(S-list(transition_matrix_SP["D"]).index(countries_PD_SP[country]),2))
EAD_vector = np.array(EAD_vector)/np.sum(EAD_vector)
ELGD_vector = np.array([0.45]*len(EAD_vector))
PD_vector = np.array(PD_vector)
q=0.999
def to_solve(om):
    b = (0.11852-0.05478*np.log(PD_vector))**2
    R_corr = 0.12*(1-np.exp(-50*PD_vector))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD_vector))/(1-np.exp(-50)))
    K = ((ELGD_vector)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD_vector)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD_vector*ELGD_vector)
    alpha_X =  gamma.ppf(q,a =0.25, scale = 1/0.25)
    return  (np.abs(ELGD_vector*PD_vector*om*(alpha_X-1)-K))
omega_vector = fsolve(to_solve,[0.25]*len(EAD_vector))
portfolios_0999['IDB_EEA_B-'] = [EAD_vector,ELGD_vector,PD_vector,omega_vector,g_vector]

Create Entry for IDB subject to B- Floor

In [49]:
EAD_vector,ELGD_vector,PD_vector,omega_vector,g_vector = portfolios_0999['IDB_B-']
q=0.999
def to_solve(om):
    b = (0.11852-0.05478*np.log(PD_vector))**2
    R_corr = 0.12*(1-np.exp(-50*PD_vector))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD_vector))/(1-np.exp(-50)))
    K = ((ELGD_vector)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD_vector)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD_vector*ELGD_vector)
    alpha_X =  gamma.ppf(q,a =0.25, scale = 1/0.25)
    return  (np.abs(ELGD_vector*PD_vector*om*(alpha_X-1)-K))
omega_vector = fsolve(to_solve,[0.25]*len(EAD_vector))
portfolios_0999['IDB_B-'] = [EAD_vector,ELGD_vector,PD_vector,omega_vector,g_vector]

PD Distribution

In [50]:
all_PDs = np.unique([v for v in countries_PD_SP.values()])
all_PDs_dict = {}
for PD in all_PDs:
    all_PDs_dict[np.round(PD,2)] = 0
portfolios_099_plot = create_portfolio(q = 0.99,transition_matrix = transition_matrix_SP,countries_PD = countries_PD_SP,normalize_shares = False)
Porfolio_PDs_list = []
for pf in portfolios_099_plot:
    pf_PDs = np.unique([v for v in countries_PD_SP.values()])
    pf_PDs_dict = {}
    for PD in all_PDs:
        pf_PDs_dict[np.round(PD,2)] = 0
    EAD = portfolios_099_plot[pf][0]
    PD = portfolios_099_plot[pf][2]
    for i in range(len(EAD)):        
        pf_PDs_dict[np.round(100*PD[i],2)] += EAD[i]
    pf_exposures = np.sum([v for v in pf_PDs_dict.values()])    
    for k in all_PDs_dict.keys():
        pf_PDs_dict[k] = pf_PDs_dict[k]/pf_exposures
    Porfolio_PDs_list.append(pf_PDs_dict)
PD_mean = {}
for PD in all_PDs:
    PD_mean[np.round(PD,2)] = np.mean([Porfolio_PDs_list[i][np.round(PD,2)] for i in range(len(Porfolio_PDs_list))])
PD_mean

{0.0: 0.0,
 0.01: 0.021102100995208258,
 0.02: 0.0,
 0.04: 0.031054183560633986,
 0.06: 0.21131588647253963,
 0.11: 0.0067453004054552155,
 0.18: 0.10274603759675636,
 0.4: 0.082519351271655,
 0.9: 0.19104312569111684,
 1.46: 0.029294139329155915,
 2.38: 0.021341688168079617,
 7.59: 0.11370254330998894,
 51.47: 0.18913564319941026}

In [51]:
import json
print("Started writing dictionary to a file")
with open("probs.txt", "w") as fp:
    json.dump(PD_mean, fp)  # encode dict into JSON
print("Done writing dict into .txt file")

Started writing dictionary to a file
Done writing dict into .txt file


# Test everything

Define a Function for the computations of the granularity adjustment (GA)

In [52]:
def compute_GA(q=0.999,ELGD_const = 0.45,
               portfolios = portfolios_0999,
               trans_dict = SP_dict,
               compute_mtm=True,
               constant_rho = False,
               constant_rho_val = 0.35,
               relative_to_UL = False,
              n_mc = 1000000, # For Actuarial Approach
              n_mc_mtm = 500000,
              n_mc_mtm_n1 = 50,
              maturity = np.array([1]*len(portfolios_0999)),
              xi = 0.25,
              coupons = 0.01):
    GA_approx = []
    GA_approx_simplified_LGD_const = []
    GA_approx_simplified = []
    GA_approx_VLGD = []
    GA_approx_2nd = []
    GA_approx_omegas = []
    GA_IRB_MC_list = []
    GA_IRB_MC_LGD_const_list = []
    GA_MC = []
    GA_MC_LGD_const = []
    GA_MC_omegas = []
    GA_MC_LGD_const_omegas = []
    GA_MTM_DO = []
    GA_MTM_DO_LGD_const = []
    GA_MTM_approx_DO = []
    GA_MTM_approx = []
    GA_MTM_approx_DO_LGD_const = []
    GA_MTM_approx_LGD_const = []
    GA_MTM = []
    GA_MTM_LGD_const = []
    
    i = -1
    for portfolio in portfolios:
        i = i +1
        EAD,ELGD,PD,omega,g = portfolios[portfolio]
        ELGD = np.array([ELGD_const]*len(EAD))
        if len(EAD)>100: # Choose 100 indices
            indices = np.random.choice(range(len(EAD)-1), 100,replace = False)
            EAD = EAD[indices]
            PD = PD[indices]
            omega = omega[indices]
            ELGD = ELGD[indices]
            g = np.array(g)[indices]
        EAD = EAD/np.sum(EAD)
        if constant_rho == False:
            if relative_to_UL:
                # Determine Kstar
                def to_solve(R_corr):
                    K = ((ELGD)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD*ELGD)
                    alpha_X =  gamma.ppf(q,a =0.25, scale = 1/0.25)
                    return ELGD*PD*omega*(alpha_X-1)-K
                R_corr = fsolve(to_solve,[0.18]*len(EAD))
                K = ((ELGD)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD*ELGD)
                K_star_omega = np.sum(K*EAD)
                R_corr = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50)))
                K = ((ELGD)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD*ELGD)
                K_star_IRB = np.sum(K*EAD)
            if compute_mtm:
                GA_MTM.append(MC_IS_MtM(ELGD,EAD,
                                            rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50))),
                                            c = np.array([coupons]*len(EAD)),
                                            g = np.array(g),
                                            D= np.array([maturity[i]]*len(EAD)),                                        
                                            trans_dict = trans_dict,
                                            r= r_nelson,
                                            q=q,T=1,n = n_mc_mtm,n1  = n_mc_mtm_n1,default_only=False))
                GA_MTM_LGD_const.append(MC_IS_MtM(ELGD,EAD,
                            rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50))),
                            c = np.array([coupons]*len(EAD)),
                            g = np.array(g),
                            D= np.array([maturity[i]]*len(EAD)),                                        
                            trans_dict = trans_dict,
                            r= r_nelson,
                            q=q,T=1,n = n_mc_mtm,n1  = n_mc_mtm_n1,default_only=False,LGD_constant = True))
                GA_MTM_approx_LGD_const.append(GA_GM(PD = PD, ELGD =ELGD, A = EAD, M= np.array([maturity[i]]*len(EAD)),
                      q=q,        
                     r = r_nelson,
                     T=1,
                     g= np.array(g),
                     trans_prob=trans_dict["trans_prob"],#N times S matrix
                     psi = 0.4, # Market Sharpe Ratio,
                     rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50))),
                     c = coupons,
                    nu = 0.0,
                     S = S, # Number of States of Rating
                     default_only = False,
                      LGD_constant = True))
                GA_MTM_approx.append(GA_GM(PD = PD, ELGD =ELGD, A = EAD, M= np.array([maturity[i]]*len(EAD)),
                      q=q,        
                     r = r_nelson,
                     T=1,
                     g= np.array(g),
                     trans_prob=trans_dict["trans_prob"],#N times S matrix
                     psi = 0.4, # Market Sharpe Ratio,
                     rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50))),
                     c = coupons,
                    nu = 0.25,
                     S = S, # Number of States of Rating
                     default_only = False,
                      LGD_constant = False))
                GA_MTM_DO.append(MC_IS_MtM(ELGD,EAD,
                                            rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50))),
                                            c = np.array([coupons]*len(EAD)),
                                            g = np.array(g),
                                            D= np.array([maturity[i]]*len(EAD)),
                                            trans_dict = trans_dict,
                                            r= r_nelson,
                                            q=q,T=1,n = n_mc_mtm,n1 = n_mc_mtm_n1,default_only=True))
                GA_MTM_DO_LGD_const.append(MC_IS_MtM(ELGD,EAD,
                                            rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50))),
                                            c = np.array([coupons]*len(EAD)),
                                            g = np.array(g),
                                            D= np.array([maturity[i]]*len(EAD)),
                                            trans_dict = trans_dict,
                                            r= r_nelson,
                                            q=q,T=1,n = n_mc_mtm,n1 = n_mc_mtm_n1,default_only=True,LGD_constant = True))
                GA_MTM_approx_DO_LGD_const.append(GA_GM(PD = PD, ELGD =ELGD, A = EAD, M= np.array([maturity[i]]*len(EAD)),
                      q=q,        
                     r = r_nelson,
                     T=1,
                     g= np.array(g),
                     trans_prob=trans_dict["trans_prob"],#N times S matrix
                     psi = 0.4, # Market Sharpe Ratio,
                     rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50))),
                     c = coupons,
                    nu =0,               
                     S = S, # Number of States of Rating
                     default_only = True,
                      LGD_constant = True))
                GA_MTM_approx_DO.append(GA_GM(PD = PD, ELGD =ELGD, A = EAD, M= np.array([maturity[i]]*len(EAD)),
                      q=q,        
                     r = r_nelson,
                     T=1,
                     g= np.array(g),
                     trans_prob=trans_dict["trans_prob"],#N times S matrix
                     psi = 0.4, # Market Sharpe Ratio,
                     rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50))),
                     c = coupons,
                    nu =0.25,               
                     S = S, # Number of States of Rating
                     default_only = True,
                      LGD_constant = False))
            
            GA_IRB_MC_list.append(GA_IRB_MC(PD, # Vector of default probabillities
                  ELGD, #Vector of expected losses given default,
                  EAD, # Vector of Exposures at default,
                  q = q, # quantile level
                  N_sim = n_mc,
                  nu = 0.25,
                  LGD_constant = False,
                  M = maturity[i]))
            
            GA_IRB_MC_LGD_const_list.append(GA_IRB_MC(PD, # Vector of default probabillities
                  ELGD, #Vector of expected losses given default,
                  EAD, # Vector of Exposures at default,
                  q = q, # quantile level
                  N_sim = n_mc,
                  LGD_constant = True,
                  M = maturity[i]))
            
            GA_approx_simplified.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  simplified = True,
                  LGD_constant = False,
                  IRB = True))
            GA_approx_simplified_LGD_const.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  simplified = True,
                  LGD_constant = True,
                  IRB = True))
            
            GA_approx_VLGD.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  LGD_constant = False,
                  IRB = True))
            GA_approx_2nd.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  LGD_constant = False,
                  second_order = True,
                  IRB = True))

            GA_approx.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  LGD_constant = True,
                  IRB = True)) 
            GA_approx_omegas.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  LGD_constant = True,
                  IRB = False))
            if relative_to_UL:
                GA_IRB_MC_list[-1] = GA_IRB_MC_list[-1]/(K_star_IRB+GA_IRB_MC_list[-1])                
                GA_IRB_MC_LGD_const_list[-1] = GA_IRB_MC_LGD_const_list[-1]/(K_star_IRB+GA_IRB_MC_LGD_const_list[-1])
                GA_approx_simplified[-1] = GA_approx_simplified[-1]/(K_star_IRB+GA_approx_simplified[-1])
                GA_approx_simplified_LGD_const[-1]   = GA_approx_simplified_LGD_const[-1]/(K_star_IRB+GA_approx_simplified_LGD_const[-1])
                GA_approx_VLGD[-1] = GA_approx_VLGD[-1]/(K_star_IRB+GA_approx_VLGD[-1])
                GA_approx[-1] = GA_approx[-1]/(K_star_IRB+GA_approx[-1])
                GA_approx_omegas[-1] = GA_approx_omegas[-1]/(K_star_omega+GA_approx_omegas[-1])
                
            # Adjust Input:
            rho = 0.12*(1-np.exp(-50*PD))/(1-np.exp(-50))+0.24*(1-(1-np.exp(-50*PD))/(1-np.exp(-50)))
            rho = np.reshape(np.append(rho,[0]*(100-len(EAD))),(1,100))
            c = np.array([coupons]*len(EAD))
            c = np.reshape(np.append(c,[0]*(100-len(EAD))),(1,100))
            g = np.array(g)
            g = np.reshape(np.append(g,[0]*(100-len(EAD))),(1,100))
            D= np.array([1]*len(EAD))
            D=  np.reshape(np.append(D,[0]*(100-len(EAD))),(1,100))
            EAD = np.reshape(np.append(EAD,[0]*(100-len(EAD))),(1,100))
            ELGD = np.reshape(np.append(ELGD,[0]*(100-len(ELGD))),(1,100))
            PD = np.reshape(np.append(PD,[0]*(100-len(PD))),(1,100))
            omega = np.reshape(np.append(omega,[0]*(100-len(omega))),(1,100))
            Input = np.concatenate([ELGD ,EAD ,PD ,omega ],axis =1)
            Input_mtm = np.concatenate([ELGD ,EAD,rho ,c, g,  D ],axis =1)
            GA_MC_LGD_const.append(MC_IS_actuarial(ELGD ,EAD ,PD ,omega,q=q,IRB = True,n = n_mc,LGD_constant = True)[0])
            GA_MC_LGD_const_omegas.append(MC_IS_actuarial(ELGD ,EAD ,PD ,omega,q=q,IRB = False,n = n_mc,LGD_constant = True)[0])
            GA_MC.append(MC_IS_actuarial(ELGD ,EAD ,PD ,omega,q=q,IRB = True,n = n_mc)[0])
            GA_MC_omegas.append(MC_IS_actuarial(ELGD ,EAD ,PD ,omega,q=q,IRB = False,n = n_mc)[0])
            if relative_to_UL:
                GA_MC[-1] =  GA_MC[-1]/(K_star_IRB+GA_MC[-1] )
                GA_MC_omegas[-1] =  GA_MC_omegas[-1]/(K_star_omega+GA_MC_omegas[-1])
                GA_MC_LGD_const[-1] =  GA_MC_LGD_const[-1]/(K_star_IRB+GA_MC_LGD_const[-1] )
                GA_MC_LGD_const_omegas[-1] =  GA_MC_LGD_const_omegas[-1]/(K_star_omega+GA_MC_LGD_const_omegas[-1])
            if compute_mtm:
                df = pd.DataFrame([[len(portfolios[portfolio][0])/100 for portfolio in portfolios],
                                                    GA_approx,GA_approx_omegas,
                                      GA_approx_VLGD,GA_approx_2nd, GA_approx_simplified,GA_approx_simplified_LGD_const,
                                      GA_MC,GA_MC_omegas,GA_MC_LGD_const,GA_MC_LGD_const_omegas,
                                   GA_IRB_MC_list,GA_IRB_MC_LGD_const_list,
                                      GA_MTM_DO,GA_MTM_approx_DO,GA_MTM_DO_LGD_const, GA_MTM_approx_DO_LGD_const,
                                     GA_MTM,GA_MTM_approx,GA_MTM_LGD_const, GA_MTM_approx_LGD_const], 
                                     columns=portfolios_0999.keys(), 
                                     index = ["Borrowers",
                                              "GA approx (rho from IRB, VLGD  = 0)",
                                              "GA approx (omega as Input , VLGD = 0)", 
                                              "GA approx (rho from IRB, nu = 0.25)",
                                              "GA approx (rho from IRB, nu = 0.25, 2nd order)",
                                              "GA approx (simplified, nu = 0.25)",
                                              "GA approx (simplified, nu = 0)",
                                              "GA Monte Carlo (CR+, rho from IRB, nu = 0.25)",
                                             "GA Monte Carlo (CR+, omega as Input, nu = 0.25)",
                                             "GA Monte Carlo (CR+, rho from IRB, LGD constant)",
                                             "GA Monte Carlo (CR+, omega as Input, LGD constant)",
                                            "GA Monte Carlo (IRB, nu = 0.25)",
                                            "GA Monte Carlo (IRB, LGD constant)",
                                             "GA MTM Monte Carlo (Default Only, nu = 0.25)",
                                             "GA MTM approx. (Default Only, nu = 0.25)",
                                             "GA MTM Monte Carlo (Default Only, nu = 0)",
                                             "GA MTM Approx. (Default Only , nu = 0)",
                                            "GA MTM Monte Carlo, nu = 0.25",
                                            "GA MTM Approx. , nu = 0.25",
                                            "GA MTM Monte Carlo, nu = 0",
                                             "GA MTM Approx. , nu = 0"])
            else:
                df = pd.DataFrame([[len(portfolios[portfolio][0])/100 for portfolio in portfolios],
                                                    GA_approx,GA_approx_omegas,
                                      GA_approx_VLGD,GA_approx_2nd,GA_approx_simplified,GA_approx_simplified_LGD_const,
                                      GA_MC,GA_MC_omegas,GA_MC_LGD_const,GA_MC_LGD_const_omegas,
                                   GA_IRB_MC_list,GA_IRB_MC_LGD_const_list], 
                                     columns=portfolios_0999.keys(), 
                                     index = ["Borrowers",
                                              "GA approx (rho from IRB, VLGD  = 0)",
                                              "GA approx (omega as Input , VLGD = 0)", 
                                              "GA approx (rho from IRB, nu = 0.25)",
                                              "GA approx (rho from IRB, nu = 0.25, 2nd order)",
                                              "GA approx (simplified, nu = 0.25)",                                              
                                              "GA approx (simplified, nu = 0)",
                                              "GA Monte Carlo (CR+, rho from IRB, nu = 0.25)",
                                             "GA Monte Carlo (CR+, omega as Input, nu = 0.25)",
                                             "GA Monte Carlo (CR+, rho from IRB, LGD constant)",
                                             "GA Monte Carlo (CR+, omega as Input, LGD constant)",
                                            "GA Monte Carlo (IRB model, nu = 0.25)",
                                            "GA Monte Carlo (IRB model,  LGD constant)"])

                
                
                
                
        elif constant_rho == True:
            if relative_to_UL:
                R_corr = np.array([constant_rho_val]*len(EAD))
                K = ((ELGD)*norm.cdf(np.sqrt(1/(1-R_corr))*norm.ppf(PD)+np.sqrt(R_corr/(1-R_corr))*norm.ppf(q))-PD*ELGD)
                K_star = np.sum(K*EAD)
            if compute_mtm:
                GA_MTM.append(MC_IS_MtM(ELGD,EAD,
                                            rho = np.array([constant_rho_val]*len(EAD)),
                                            c = np.array([coupons]*len(EAD)),
                                            g = np.array(g),
                                            D= np.array([maturity[i]]*len(EAD)),                                        
                                            trans_dict = trans_dict,
                                            r= r_nelson,
                                            q=q,T=1,n = n_mc_mtm,n1 = n_mc_mtm_n1,default_only=False))
                GA_MTM_LGD_const.append(MC_IS_MtM(ELGD,EAD,
                                rho = np.array([constant_rho_val]*len(EAD)),
                                c = np.array([coupons]*len(EAD)),
                                g = np.array(g),
                                D= np.array([maturity[i]]*len(EAD)),                                        
                                trans_dict = trans_dict,
                                r= r_nelson,
                                q=q,T=1,n = n_mc_mtm,n1 = n_mc_mtm_n1,default_only=False,LGD_constant = True)) 
                GA_MTM_approx_LGD_const.append(GA_GM(PD = PD, ELGD =ELGD, A = EAD, M= np.array([maturity[i]]*len(EAD)),
                      q=q,        
                     r = r_nelson,
                     T=1,
                     g= np.array(g),
                     trans_prob=trans_dict["trans_prob"],#N times S matrix
                     psi = 0.4, # Market Sharpe Ratio,
                     rho = np.array([constant_rho_val]*len(EAD)),
                     c = coupons,                                    
                     S = S, # Number of States of Rating
                     default_only = False,
                      LGD_constant = True))
                GA_MTM_approx.append(GA_GM(PD = PD, ELGD =ELGD, A = EAD, M= np.array([maturity[i]]*len(EAD)),
                      q=q,        
                     r = r_nelson,
                     T=1,
                     g= np.array(g),
                     trans_prob=trans_dict["trans_prob"],#N times S matrix
                     psi = 0.4, # Market Sharpe Ratio,
                     rho = np.array([constant_rho_val]*len(EAD)),
                     nu =0.25,
                     c = coupons,                                    
                     S = S, # Number of States of Rating
                     default_only = False,
                      LGD_constant = False))
                GA_MTM_DO.append(MC_IS_MtM(ELGD,EAD,
                                            rho = np.array([constant_rho_val]*len(EAD)),
                                            c = np.array([coupons]*len(EAD)),
                                            g = np.array(g),
                                            D= np.array([maturity[i]]*len(EAD)),                                           
                                            trans_dict = trans_dict,
                                            r= r_nelson,
                                            q=q,T=1,n = n_mc_mtm,n1 = n_mc_mtm_n1,default_only=True))
                GA_MTM_DO_LGD_const.append(MC_IS_MtM(ELGD,EAD,
                                rho = np.array([constant_rho_val]*len(EAD)),
                                c = np.array([coupons]*len(EAD)),
                                g = np.array(g),
                                D= np.array([maturity[i]]*len(EAD)),                                           
                                trans_dict = trans_dict,
                                r= r_nelson,
                                q=q,T=1,n = n_mc_mtm,n1 = n_mc_mtm_n1,default_only=True,LGD_constant = True))
                GA_MTM_approx_DO.append(GA_GM(PD = PD, ELGD =ELGD, A = EAD, M= np.array([maturity[i]]*len(EAD)),
                      q=q,        
                     r = r_nelson,
                     T=1,
                     g= np.array(g),
                     trans_prob=trans_dict["trans_prob"],#N times S matrix
                     psi = 0.4, # Market Sharpe Ratio,
                     rho = np.array([constant_rho_val]*len(EAD)),
                     c = coupons,
                     S = S, # Number of States of Rating
                     default_only = True,
                      LGD_constant = True))
                GA_MTM_approx_DO_LGD_const.append(GA_GM(PD = PD, ELGD =ELGD, A = EAD, M= np.array([maturity[i]]*len(EAD)),
                      q=q,        
                     r = r_nelson,
                     T=1,
                     g= np.array(g),
                     trans_prob=trans_dict["trans_prob"],#N times S matrix
                     psi = 0.4, # Market Sharpe Ratio,
                     rho = np.array([constant_rho_val]*len(EAD)),
                     c = coupons,
                     nu = 0.25,
                     S = S, # Number of States of Rating
                     default_only = True,
                      LGD_constant = False))
            GA_IRB_MC_list.append(GA_IRB_MC(PD, # Vector of default probabillities
                  ELGD, #Vector of expected losses given default,
                  EAD, # Vector of Exposures at default,
                  q = q, # quantile level
                  N_sim = n_mc,
                  constant_rho = True,
                  constant_rho_val = constant_rho_val,
                  M = maturity[i]))
            GA_IRB_MC_LGD_const_list.append(GA_IRB_MC(PD, # Vector of default probabillities
                  ELGD, #Vector of expected losses given default,
                  EAD, # Vector of Exposures at default,
                  q = q, # quantile level
                  N_sim = n_mc,
                  constant_rho = True,
                  constant_rho_val = constant_rho_val,
                  nu = 0.25,
                  LGD_constant = True,
                  M = maturity[i]))
            GA_approx_simplified.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  simplified = True,
                  LGD_constant = False,
                  IRB = True,
                  constant_rho = True,
                  constant_rho_val = constant_rho_val))
            GA_approx_simplified_LGD_const.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  simplified = True,
                  LGD_constant = True,
                  IRB = True,
                  constant_rho = True,
                  constant_rho_val = constant_rho_val))

            GA_approx.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  LGD_constant = True,
                  IRB = True,
                  constant_rho = True,
                  constant_rho_val = constant_rho_val)) 
    
            GA_approx_omegas.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  LGD_constant = True,
                  IRB = False,
                  constant_rho = False,
                  constant_rho_val = constant_rho_val)) 
            GA_approx_VLGD.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  LGD_constant = False,
                  IRB = True,
                  constant_rho = True,
                  constant_rho_val = constant_rho_val))
            GA_approx_2nd.append(GA_GL(PD,
                                           ELGD,
                                           EAD, maturity[i], # Vector of Maturities
                  q = q, # quantile level
                  xi = xi, # precision parameter,
                  nu = 0.25,# recovery parameter
                  rho = omega,
                  LGD_constant = False,
                  second_order = True,
                  IRB = True,
                  constant_rho = True,
                  constant_rho_val = constant_rho_val))
            if relative_to_UL:
                GA_IRB_MC_list[-1] = GA_IRB_MC_list[-1]/(K_star+GA_IRB_MC_list[-1])
                GA_IRB_MC_LGD_const_list[-1] = GA_IRB_MC_LGD_const_list[-1]/(K_star+GA_IRB_MC_LGD_const_list[-1])
                GA_approx_simplified[-1] = GA_approx_simplified[-1]/(K_star+GA_approx_simplified[-1])
                GA_approx_simplified_LGD_const[-1] = GA_approx_simplified_LGD_const[-1]/(K_star+GA_approx_simplified_LGD_const[-1])
                GA_approx[-1] = GA_approx[-1]/(K_star+GA_approx[-1])
                GA_approx_omegas[-1] = GA_approx_omegas[-1]/(K_star+GA_approx_omegas[-1])
                GA_approx_VLGD[-1] =  GA_approx_VLGD[-1]/(K_star+GA_approx_VLGD[-1])
                GA_approx_2nd[-1] = GA_approx_2nd[-1]/(K_star+GA_approx_2nd[-1])
            # Adjust Input for NNs:
            rho = np.array([constant_rho_val]*len(EAD))
            rho = np.reshape(np.append(rho,[0]*(100-len(EAD))),(1,100))
            c = np.array([coupons]*len(EAD))
            c = np.reshape(np.append(c,[0]*(100-len(EAD))),(1,100))
            g = np.array(g)
            g = np.reshape(np.append(g,[0]*(100-len(EAD))),(1,100))
            D= np.array([1]*len(EAD))
            D=  np.reshape(np.append(D,[0]*(100-len(EAD))),(1,100))
            EAD = np.reshape(np.append(EAD,[0]*(100-len(EAD))),(1,100))
            ELGD = np.reshape(np.append(ELGD,[0]*(100-len(ELGD))),(1,100))
            PD = np.reshape(np.append(PD,[0]*(100-len(PD))),(1,100))
            omega = np.reshape(np.append(omega,[0]*(100-len(omega))),(1,100))
            Input = np.concatenate([ELGD ,EAD ,PD ,omega ],axis =1)
            Input_mtm = np.concatenate([ELGD ,EAD,rho ,c, g,  D ],axis =1)
            
            GA_MC_LGD_const.append(MC_IS_actuarial(ELGD ,EAD ,PD ,omega,q=q,constant_rho = True,
                  constant_rho_val = constant_rho_val,IRB = True,n = n_mc,LGD_constant = True)[0])
            GA_MC_LGD_const_omegas.append(MC_IS_actuarial(ELGD ,EAD ,PD ,omega,q=q, IRB = False,n = n_mc,LGD_constant = True)[0])
            GA_MC.append(MC_IS_actuarial(ELGD ,EAD ,PD ,omega,q=q,                  constant_rho = True,
                  constant_rho_val = constant_rho_val,IRB = True,n = n_mc)[0])
            GA_MC_omegas.append(MC_IS_actuarial(ELGD ,EAD ,PD ,omega,q=q,                  constant_rho = True,IRB = False,n = n_mc)[0])
            if relative_to_UL:
                GA_MC[-1] =  GA_MC[-1]/(K_star+GA_MC[-1] )
                GA_MC_omegas[-1] =  GA_MC_omegas[-1]/(K_star+GA_MC_omegas[-1])
                GA_MC_LGD_const[-1] =  GA_MC_LGD_const[-1]/(K_star+GA_MC_LGD_const[-1] )
                GA_MC_LGD_const_omegas[-1] =  GA_MC_LGD_const_omegas[-1]/(K_star+GA_MC_LGD_const_omegas[-1])
            if compute_mtm:
                df = pd.DataFrame([[len(portfolios[portfolio][0])/100 for portfolio in portfolios],
                                                    GA_approx,GA_approx_VLGD,GA_approx_2nd,GA_approx_omegas,
                                      GA_approx_simplified,GA_approx_simplified_LGD_const,
                                      GA_MC,GA_MC_omegas,GA_MC_LGD_const,GA_MC_LGD_const_omegas,
                                   GA_IRB_MC_list,GA_IRB_MC_LGD_const_list,
                                      GA_MTM_DO,GA_MTM_approx_DO,GA_MTM_DO_LGD_const, GA_MTM_approx_DO_LGD_const,
                                     GA_MTM,GA_MTM_approx,GA_MTM_LGD_const, GA_MTM_approx_LGD_const], 
                                     columns=portfolios_0999.keys(), 
                                     index = ["Borrowers",
                                              "GA approx (rho constant, nu = 0)",
                                              "GA approx (rho constant, nu = 0.25)",
                                              "GA approx (rho constant, nu = 0.25 2nd order)",
                                              "GA approx (omega as Input, nu = 0)", 
                                              "GA approx (simplified, rho constant, nu = 0.25)",
                                              "GA approx (simplified, rho constant, nu = 0)",
                                              "GA Monte Carlo (CR+, rho from IRB, nu = 0.25)",
                                             "GA Monte Carlo (CR+, omega as Input, nu = 0.25)",
                                             "GA Monte Carlo (CR+, rho from IRB, LGD constant)",
                                             "GA Monte Carlo (CR+, omega as Input, LGD constant)",
                                            "GA Monte Carlo (IRB model, rho constant, nu = 0.25)",
                                            "GA Monte Carlo (IRB model, rho constant, LGD constant)",
                                             "GA MTM Monte Carlo (Default Only, nu = 0.25)",
                                             "GA MTM approx. (Default Only, nu = 0.25)",
                                             "GA MTM Monte Carlo (Default Only, nu = 0)",
                                             "GA MTM Approx. (Default Only , nu = 0)",
                                            "GA MTM Monte Carlo, nu = 0.25",
                                            "GA MTM Approx. , nu = 0.25",
                                            "GA MTM Monte Carlo, nu = 0",
                                             "GA MTM Approx. , nu = 0"])
            else:
                df = pd.DataFrame([[len(portfolios[portfolio][0])/100 for portfolio in portfolios],
                                                    GA_approx,GA_approx_VLGD,GA_approx_2nd, GA_approx_omegas,
                                      GA_approx_simplified,GA_approx_simplified_LGD_const,
                                      GA_MC,GA_MC_omegas,GA_MC_LGD_const,GA_MC_LGD_const_omegas,
                                   GA_IRB_MC_list,GA_IRB_MC_LGD_const_list], 
                                     columns=portfolios.keys(), 
                                     index = ["Borrowers",
                                              "GA approx (rho constant, nu = 0)",
                                              "GA approx (rho constant, nu = 0.25)",
                                              "GA approx (rho constant, nu = 0.25 2nd order)",
                                              "GA approx (omega as Input, nu = 0)", 
                                              "GA approx (simplified, rho constant, nu = 0.25)",
                                              "GA approx (simplified, rho constant, nu = 0)",
                                              "GA Monte Carlo (CR+, rho from IRB, nu = 0.25)",
                                             "GA Monte Carlo (CR+, omega as Input, nu = 0.25)",
                                             "GA Monte Carlo (CR+, rho from IRB, LGD constant)",
                                             "GA Monte Carlo (CR+, omega as Input, LGD constant)",
                                            "GA Monte Carlo (IRB model, rho constant, nu = 0.25)",
                                            "GA Monte Carlo (IRB model, rho constant, LGD constant)"]) 
                
    return df*100 # Output everything in percent

# GA Computations for S&P Transition Matrix

In [53]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 1.) q = 99.9%, ELGD = 0.1

In [54]:
df_999 = compute_GA(q=0.999,ELGD_const = 0.1,
               portfolios = portfolios_0999)
display(df_999)

,IDB,IDB_B-,IDB_EEA,IDB_EEA_B-
Borrowers,26.00,26.00,44.00,44.00
"GA approx (rho from IRB, VLGD = 0)",3.61,2.09,3.13,1.78
"GA approx (omega as Input , VLGD = 0)",3.70,2.09,3.13,1.78
"GA approx (rho from IRB, nu = 0.25)",16.97,7.87,14.83,6.70
"GA approx (rho from IRB, nu = 0.25, 2nd order)",198.16,61.07,162.82,48.64
"GA approx (simplified, nu = 0.25)",11.72,6.81,10.18,5.79
"GA approx (simplified, nu = 0)",3.61,2.09,3.13,1.78
"GA Monte Carlo (CR+, rho from IRB, nu = 0.25)",3.27,2.71,2.99,2.44
"GA Monte Carlo (CR+, omega as Input, nu = 0.25)",3.27,2.69,2.98,2.45
"GA Monte Carlo (CR+, rho from IRB, LGD constant)",1.11,1.26,0.94,1.09


## 2.) q = 99.9%, ELGD = 0.2

In [55]:
df_999_02 = compute_GA(q=0.999,ELGD_const = 0.2,
               portfolios = portfolios_0999)
display(df_999_02)

,IDB,IDB_B-,IDB_EEA,IDB_EEA_B-
Borrowers,26.00,26.00,44.00,44.00
"GA approx (rho from IRB, VLGD = 0)",7.21,4.19,6.27,3.56
"GA approx (omega as Input , VLGD = 0)",7.40,4.19,6.27,3.56
"GA approx (rho from IRB, nu = 0.25)",19.10,9.32,16.66,7.94
"GA approx (rho from IRB, nu = 0.25, 2nd order)",125.34,43.90,103.29,35.16
"GA approx (simplified, nu = 0.25)",14.43,8.38,12.53,7.13
"GA approx (simplified, nu = 0)",7.21,4.19,6.27,3.56
"GA Monte Carlo (CR+, rho from IRB, nu = 0.25)",5.80,4.82,5.25,4.32
"GA Monte Carlo (CR+, omega as Input, nu = 0.25)",5.81,4.85,5.21,4.34
"GA Monte Carlo (CR+, rho from IRB, LGD constant)",2.20,2.55,1.87,2.22
